# Funktion und automatisch intern von python generierte Funktion __get__()
## jede Funktion oder Methode in einer Klasse ist ein Objekt vom Typ function
### Frage: 
Auto.fahren.__get__(mein_auto) :
- wird   ein __get__()  bei Erzeugung einer Klasse automatisch für jede in der Klasse definierten Klasse intern von python erzeugt und durch Auto.fahren.__get__(mein_auto)  wird automatisch die von User definierte originale Funktion aufgerufen?
- wird auch  ein __get__() für Funktion ausserhalb von einer Klasse automatisch intern von python erzeugt?
erkläre den Satz noch genauer:
### Antwort:
Die __get__-Methode der Funktion erzeugt "on the fly" (während des Zugriffs) ein neues Objekt: eine Bound Method (gebundene Methode)., und was ist eine gebundene Methode, zeige ein Beispiel wie der gebundene Methode oder Wrraper aussieht?

1. Wird __get__ automatisch erzeugt?
    - Ja. In Python ist jede Funktion (egal ob innerhalb oder außerhalb einer Klasse definiert) ein Objekt vom Typ function. Die Klasse function besitzt von Haus aus eine eingebaute __get__-Methode.

    - Du musst als Programmierer nichts tun. Sobald du def name(): schreibst, hat dieses Objekt automatisch die Fähigkeit, als Descriptor zu fungieren.
    - Wichtig: Es wird nicht für jede Funktion eine neue __get__-Logik geschrieben, sondern alle Funktionen teilen sich die __get__-Implementierung der Basis-Klasse function.

2. Funktionen außerhalb von Klassen
    - Ja, auch Funktionen außerhalb von Klassen haben ein __get__. Du kannst es sogar manuell aufrufen, auch wenn das im Alltag niemand tut:

Der Unterschied ist: Außerhalb einer Klasse wird __get__ beim normalen Aufruf einfach nicht benutzt. Es kommt erst zum Einsatz, wenn die Funktion als Attribut einer Klasse gefunden wird.


### No Data descriptor: nur mit __get__()

In [15]:
class Auto:
    def fahren(self):
        print( "running fahren")
        return " fahren Done"
        
print("-1 ", Auto.fahren.__get__ )

a = Auto()
print("0 ", Auto.fahren.__get__(a) )
print("1. ", Auto.__dict__)
# Zugriff über die Klasse liefert die nackte Funktion
print("2 ", Auto.fahren)  
# <function Auto.fahren at 0x...>

# Zugriff über die Instanz liefert eine "bound method"
print("3 ", a.fahren)     
# <bound method Auto.fahren of <__main__.Auto object at 0x...>>
print("4 ", type(a.fahren) ) 
print("5 ", a.fahren() ) 
# Diese beiden sind NICHT das gleiche Objekt! 
# Die bound method wurde erst beim Punkt-Zugriff erstellt.


-1  <method-wrapper '__get__' of function object at 0x0000012CFEAEC9A0>
0  <bound method Auto.fahren of <__main__.Auto object at 0x0000012CFF1F9820>>
1.  {'__module__': '__main__', 'fahren': <function Auto.fahren at 0x0000012CFEAEC9A0>, '__dict__': <attribute '__dict__' of 'Auto' objects>, '__weakref__': <attribute '__weakref__' of 'Auto' objects>, '__doc__': None}
2  <function Auto.fahren at 0x0000012CFEAEC9A0>
3  <bound method Auto.fahren of <__main__.Auto object at 0x0000012CFF1F9820>>
4  <class 'method'>
running fahren
5   fahren Done


In [7]:
def hallo(name):
    print(f"Hallo {name}")

# Auch eine globale Funktion hat __get__
print(hallo.__get__) # <method-wrapper '__get__' of function object...>
print(hallo.__get__("Chen")) # Aufruf von methode
hallo("Chen")

<method-wrapper '__get__' of function object at 0x0000015C3DD02160>
<bound method hallo of 'Chen'>
Hallo Chen


# Was passiert bei meth = mein_auto.fahren?

    - Python findet fahren in der Klasse Auto.
    - Python sieht, dass fahren eine Funktion ist und __get__ hat.
    - Python führt aus: meth = Auto.fahren.__get__(mein_auto, Auto).
    - Das Ergebnis meth ist nun eine Bound Method

## Was passiert bei meth(100)?

    - Die Bound Method nimmt das Argument 100.
    - Sie schaut intern nach: "Wer ist meine Funktion? Auto.fahren. Wer ist mein Objekt? mein_auto."
    - Sie ruft auf: Auto.fahren(mein_auto, 100).

Das ist der Grund, warum wir self nicht selbst übergeben müssen – der Wrapper macht das für uns!

In [9]:
class BoundMethod:
    def __init__(self, func, obj):
        self.__func__ = func  # Die nackte Funktion aus der Klasse
        self.__self__ = obj   # Die Instanz (z.B. mein_auto)

    def __call__(self, *args, **kwargs):
        # Wenn man die Methode mit () aufruft, passiert das:
        return self.__func__(self.__self__, *args, **kwargs)


In [10]:
class Auto:
    def fahren(self, geschwindigkeit):
        print(f"Fahre {geschwindigkeit} km/h")

mein_auto = Auto()


In [2]:
class Auto:
    def fahren(self):
        pass

a = Auto()
meth = a.fahren

print(type(meth)) 
# <class 'method'>


<class 'method'>


In [4]:
print(Auto.__dict__)
print(a.fahren)

{'__module__': '__main__', 'fahren': <function Auto.fahren at 0x00000158D857D440>, '__dict__': <attribute '__dict__' of 'Auto' objects>, '__weakref__': <attribute '__weakref__' of 'Auto' objects>, '__doc__': None}
<bound method Auto.fahren of <__main__.Auto object at 0x00000158D8B75910>>
